eFL-Boost: Efficient Federated Learning for GBDT (Yamamoto, Ozawa & Wang, IEEE Access 2022)

Horizontal federated GBDT in which **the tree structure is decided locally by one client (the Builder)** and **the leaf
weights are computed globally** from the gradients of all clients. Each boosting round needs three communications:

1. **Builder → clients:** the Builder grows the trees on its own data and sends their structure (splits only, no leaf values).
2. **Clients → aggregator:** every client routes its own samples through the trees and sends the sum of gradients and
   Hessians in each leaf.
3. **Aggregator → clients:** leaf values `w = −ΣG / (ΣH + λ) · η` are computed from the summed statistics and sent back to everyone.

Multi-class uses softmax, with one tree per class per round, as in FedTree.ipynb. The histogram, split and tree code is
reused unchanged from FedTree.ipynb.

### How this implementation maps to the paper (Algorithm 1)

| Paper (Sec. III, Algorithm 1) | This notebook |
|---|---|
| Line 3: the Aggregator selects the Builder `B`, "based on a **predetermined order**", a different one for each update | `builder_rule="round_robin"`: client `t mod M` builds in round `t`. Clients with fewer than 2 samples are skipped, which can happen with Dirichlet splits. |
| Lines 4–6: every owner updates `g_d`, `h_d` from the previous trees (global model) | Softmax gradients and Hessians of each client's own samples under the global model |
| Line 7: `B` computes the tree structure from `X_B, g_B, h_B` only, using the split score of Eq. (4) | FedTree's histogram builder with a single client (the Builder). Cut points come from the Builder's own data. |
| Line 8: `B` sends the structure to the owners (**communication 1**) | Structure only (leaf values zeroed) sent to the other M−1 clients |
| Lines 9–11: each owner sends the per-leaf sums `G_d`, `H_d` to the Aggregator (**communication 2**) | Every client, including the Builder, routes its samples and sends per-leaf `ΣG`, `ΣH` |
| Lines 13–15: the Aggregator sums them, computes `w = −G/(H+λ)` (Eq. 3) and sends `w` to the owners (**communication 3**) | Same; the leaf values go to all M clients |
| Line 17: each owner combines structure + `w` into the new tree | Same |
| Table 2: 3 communications per update | `n_comm_rounds` = 3 × boosting rounds = 150 |

**Where this notebook differs from the paper (all intentional):**

* **Multi-class.** The paper only evaluates binary tasks (Table 5). Here each update builds K trees (softmax, one per class) from
  the same gradients, and the 3 communications carry all K trees together.
* **Learning rate.** Algorithm 1 uses `w = −G/(H+λ)` with no shrinkage. Here `η = 0.1` multiplies the leaf weights, as in FedTree.ipynb.
* **Split score.** Eq. (4) has no ½ factor and no γ. FedTree's `split_gain` uses ½·(…) − γ with γ = 0.001. The ½ does not
  change which split wins; γ only stops splits with almost zero gain.
* **Minimum data per leaf.** The paper recommends a predetermined lower bound on the number of samples in a leaf as a privacy
  safeguard (Secs. III-A, IV-C, V-E, Fig. 2: bounds 0–30 cost little accuracy on large datasets). It is not applied here
  (FedTree uses `min_child_weight = 0`), so leaves may contain very few samples.
* **Hyperparameters and evaluation.** The paper does not state depth, bins or learning rate, and it evaluates with 5-fold
  cross-validation using F1, AUC and log loss. Here the FedTree.ipynb settings are used (depth 6, 64 bins, η = 0.1, λ = 0.1,
  50 rounds, seed 42) with one stratified 80/20 split.
* **Communication size** is measured as in FedTree, with `sizeof_mb` (pickled payload size), for every message actually sent.

# Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict, Counter
import time
import sys
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
import copy

# Utility funcs (from FedTree.ipynb)

In [ ]:
def sizeof_mb(obj: Any) -> float:
    return sys.getsizeof(pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)) / (1024 * 1024)

def measure_inference_time(predict_fn, X, n_runs: int = 5) -> dict:
    """Measure total inference time and time-per-sample (median of several runs)."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = predict_fn(X)
        times.append(time.perf_counter() - t0)
    total = float(np.median(times))
    per_sample = total / max(len(X), 1) * 1000  # milliseconds
    return {
        "test_time_total_s": total,
        "test_time_per_sample_ms": per_sample,
    }


def softmax(scores: np.ndarray) -> np.ndarray:
    scores = scores - scores.max(axis=1, keepdims=True)
    exp = np.exp(scores)
    return exp / exp.sum(axis=1, keepdims=True)

# Cut points (from FedTree.ipynb)

In [ ]:
@dataclass
class CutPoints:
    boundaries: Dict[int, np.ndarray]   # feature -> sorted cut points

    @staticmethod
    def compute(X: np.ndarray, n_bins: int = 32) -> "CutPoints":
        boundaries = {}
        quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
        for f in range(X.shape[1]):
            col = X[:, f]
            cuts = np.unique(np.quantile(col, quantiles))
            if len(cuts) == 0:
                cuts = np.array([np.median(col)])
            boundaries[f] = cuts
        return CutPoints(boundaries=boundaries)

# Histogram helpers (from FedTree.ipynb)

In [ ]:
def compute_local_histogram(
    X: np.ndarray, g: np.ndarray, h: np.ndarray,
    instance_ids: np.ndarray, cuts: CutPoints
) -> Dict[int, np.ndarray]:
    """Returns {feat: array of shape (n_bins+1, 2)}  [sum_g, sum_h]"""
    hist = {}
    if len(instance_ids) == 0:
        for f, bnds in cuts.boundaries.items():
            hist[f] = np.zeros((len(bnds) + 1, 2), dtype=np.float64)
        return hist

    Xn = X[instance_ids]
    gn = g[instance_ids]
    hn = h[instance_ids]

    for f, bnds in cuts.boundaries.items():
        bin_idx = np.searchsorted(bnds, Xn[:, f], side="left")   # bin i  <=>  x <= bnds[i]
        n_bins = len(bnds) + 1
        gh = np.zeros((n_bins, 2), dtype=np.float64)
        np.add.at(gh[:, 0], bin_idx, gn)
        np.add.at(gh[:, 1], bin_idx, hn)
        hist[f] = gh
    return hist


def aggregate_histograms(client_hists: List[Dict[int, np.ndarray]]) -> Dict[int, np.ndarray]:
    agg = {f: np.zeros_like(h) for f, h in client_hists[0].items()}
    for hist in client_hists:
        for f, gh in hist.items():
            agg[f] += gh
    return agg

# Split gain & leaf value (from FedTree.ipynb)

In [ ]:
def leaf_value(G: float, H: float, lam: float) -> float:
    return -G / (H + lam)


def split_gain(GL, HL, GR, HR, lam, gamma) -> float:
    def term(G, H):
        return (G * G) / (H + lam) if (H + lam) > 1e-12 else 0.0
    return 0.5 * (term(GL, HL) + term(GR, HR) - term(GL + GR, HL + HR)) - gamma


def find_best_split(agg_hist: Dict[int, np.ndarray], cuts: CutPoints,
                    lam: float, gamma: float, min_child_weight: float = 1.0
                    ) -> Optional[Tuple[int, float, float]]:
    best_gain, best_feat, best_cut = -np.inf, None, None

    for f, gh in agg_hist.items():
        bnds = cuts.boundaries[f]
        cum_g = np.cumsum(gh[:, 0])
        cum_h = np.cumsum(gh[:, 1])
        G_tot, H_tot = cum_g[-1], cum_h[-1]

        for i in range(len(bnds)):
            GL, HL = cum_g[i], cum_h[i]
            GR, HR = G_tot - GL, H_tot - HL
            if HL < min_child_weight or HR < min_child_weight:
                continue
            gain = split_gain(GL, HL, GR, HR, lam, gamma)
            if gain > best_gain:
                best_gain = gain
                best_feat = f
                best_cut = bnds[i]

    if best_feat is None or best_gain <= 0:
        return None
    return best_feat, best_cut, best_gain

# Tree node & prediction (from FedTree.ipynb)

In [ ]:
@dataclass
class TreeNode:
    is_leaf: bool = False
    feature: Optional[int] = None
    threshold: Optional[float] = None
    value: float = 0.0
    left: Optional["TreeNode"] = None
    right: Optional["TreeNode"] = None


def predict_tree(node: TreeNode, X: np.ndarray) -> np.ndarray:
    out = np.zeros(X.shape[0])
    def _rec(n, idx):
        if n.is_leaf or len(idx) == 0:
            out[idx] = n.value
            return
        go_left = X[idx, n.feature] <= n.threshold
        _rec(n.left, idx[go_left])
        _rec(n.right, idx[~go_left])
    _rec(node, np.arange(X.shape[0]))
    return out

# Tree building (from FedTree.ipynb)

eFL-Boost's Builder calls this with a **single client** (itself), so the structure comes from its own data only.
The leaf values it computes locally are discarded and replaced by the global ones.

In [ ]:
def build_one_federated_tree(
    client_X: Dict[int, np.ndarray],
    client_g: Dict[int, np.ndarray],
    client_h: Dict[int, np.ndarray],
    cuts: CutPoints,
    max_depth: int,
    lam: float,
    gamma: float,
    learning_rate: float,
    min_child_weight: float,
    comm_tracker: Dict[str, float],
) -> TreeNode:

    client_ids = list(client_X.keys())
    client_inst = {cid: np.arange(client_X[cid].shape[0]) for cid in client_ids}

    root = TreeNode()
    frontier = [(root, client_inst, 0)]   # (node, {cid: ids}, depth)

    while frontier:
        node, inst_map, depth = frontier.pop(0)

        # --- PartyComputeHistogram ---
        client_hists = []
        for cid in client_ids:
            hist = compute_local_histogram(
                client_X[cid], client_g[cid], client_h[cid],
                inst_map[cid], cuts
            )
            client_hists.append(hist)
            comm_tracker["upload_mb"] += sizeof_mb(hist)

        # --- ServerAggregate ---
        agg = aggregate_histograms(client_hists)

        # Correct total G / H (any feature is fine)
        any_f = next(iter(agg))
        G_total = agg[any_f][:, 0].sum()
        H_total = agg[any_f][:, 1].sum()

        can_split = depth < max_depth
        split = find_best_split(agg, cuts, lam, gamma, min_child_weight) if can_split else None

        if split is None:
            node.is_leaf = True
            node.value = leaf_value(G_total, H_total, lam) * learning_rate
            continue

        feat, thresh, _ = split
        node.feature = feat
        node.threshold = thresh
        node.left = TreeNode()
        node.right = TreeNode()
        comm_tracker["download_mb"] += sizeof_mb((feat, thresh))

        # --- PartyUpdate: split instance sets ---
        next_left, next_right = {}, {}
        for cid in client_ids:
            ids = inst_map[cid]
            if len(ids) == 0:
                next_left[cid] = ids
                next_right[cid] = ids
                continue
            vals = client_X[cid][ids, feat]
            mask = vals <= thresh
            next_left[cid] = ids[mask]
            next_right[cid] = ids[~mask]

        frontier.append((node.left, next_left, depth + 1))
        frontier.append((node.right, next_right, depth + 1))

    return root

# Partitioning (from FedTree.ipynb)

In [ ]:
def partition_iid(X: np.ndarray, y: np.ndarray, n_clients: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    splits = np.array_split(idx, n_clients)
    return {i: (X[s], y[s]) for i, s in enumerate(splits)}


def partition_dirichlet(X: np.ndarray, y: np.ndarray, n_clients: int,
                        alpha: float = 0.5, seed: int = 42):
    rng = np.random.default_rng(seed)
    labels = np.unique(y)
    client_idx = defaultdict(list)

    for c in labels:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet([alpha] * n_clients)
        proportions = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, proportions)
        for i, s in enumerate(splits):
            client_idx[i].extend(s)

    return {i: (X[idxs], y[idxs]) for i, idxs in client_idx.items()}

# eFL-Boost helpers

* **Structure only:** a copy of the tree with every leaf value zeroed, which is what the Builder sends.
* **Routing:** each client finds the leaf index of each of its samples. Leaves are numbered in the same depth-first order everywhere.
* **Builder rule:** chooses which client builds in a given round.

In [ ]:
def collect_leaves(node: TreeNode) -> List[TreeNode]:
    """Leaves in depth-first (left-to-right) order; this order indexes the leaf statistics."""
    if node.is_leaf:
        return [node]
    return collect_leaves(node.left) + collect_leaves(node.right)


def count_nodes(node: TreeNode) -> int:
    return 1 if node.is_leaf else 1 + count_nodes(node.left) + count_nodes(node.right)


def structure_only(node: TreeNode) -> TreeNode:
    """Copy of the tree with leaf values removed: what the Builder shares."""
    tree = copy.deepcopy(node)
    for leaf in collect_leaves(tree):
        leaf.value = 0.0
    return tree


def route_to_leaves(node: TreeNode, X: np.ndarray) -> np.ndarray:
    """Leaf index (in collect_leaves order) of every row of X."""
    out = np.zeros(X.shape[0], dtype=np.int64)
    counter = [0]
    def _rec(n, idx):
        if n.is_leaf:
            out[idx] = counter[0]
            counter[0] += 1
            return
        go_left = X[idx, n.feature] <= n.threshold
        _rec(n.left, idx[go_left])
        _rec(n.right, idx[~go_left])
    _rec(node, np.arange(X.shape[0]))
    return out


def choose_builder(t: int, client_ids: List[int], sizes: Dict[int, int], rule: str,
                   rng: np.random.Generator, min_samples: int) -> int:
    eligible = [c for c in client_ids if sizes[c] >= min_samples]
    if rule == "round_robin":
        return eligible[t % len(eligible)]
    if rule == "random":
        return int(rng.choice(eligible))
    raise ValueError(rule)

# eFL-Boost

`fit` simulates all clients in one process. Each client only touches its own `X`, `y` and scores. Everything that crosses
between clients goes through `_send`, which counts the message and its size. `comm_rounds` is incremented once per
communication step, as it happens.

In [ ]:
@dataclass
class EFLBoostConfig:
    n_trees: int = 50                 # boosting rounds
    max_depth: int = 6
    n_bins: int = 64
    learning_rate: float = 0.1
    lam: float = 0.1
    gamma: float = 0.001
    min_child_weight: float = 0
    random_state: int = 42
    builder_rule: str = "round_robin"   # "round_robin" | "random"
    min_builder_samples: int = 2


class EFLBoost:
    def __init__(self, cfg: EFLBoostConfig):
        self.cfg = cfg
        self.trees: List[List[TreeNode]] = []   # [boosting_round][class]
        self.classes_: np.ndarray = None
        self.builders: List[int] = []
        self.comm_rounds = 0
        self.comm_mb = {"structure_mb": 0.0, "leaf_stats_mb": 0.0, "leaf_values_mb": 0.0}
        self.n_messages = 0

    def _send(self, payload, kind: str, n_receivers: int = 1):
        self.comm_mb[kind] += sizeof_mb(payload) * n_receivers
        self.n_messages += n_receivers
        return payload

    def fit(self, client_data: Dict[int, Tuple[np.ndarray, np.ndarray]]):
        cfg = self.cfg
        rng = np.random.default_rng(cfg.random_state)
        client_ids = sorted(client_data.keys())
        client_X = {cid: client_data[cid][0] for cid in client_ids}
        client_y = {cid: client_data[cid][1] for cid in client_ids}
        sizes = {cid: len(client_y[cid]) for cid in client_ids}
        M = len(client_ids)

        self.classes_ = np.unique(np.concatenate([client_y[c] for c in client_ids]))
        K = len(self.classes_)
        class_to_idx = {c: i for i, c in enumerate(self.classes_)}
        client_yi = {cid: np.array([class_to_idx[v] for v in client_y[cid]], dtype=np.int64) for cid in client_ids}
        client_score = {cid: np.zeros((sizes[cid], K)) for cid in client_ids}
        local_cuts = {}                     # each client's own cut points, computed when it first builds

        for t in range(cfg.n_trees):
            # ---- local: every client computes softmax gradients of its own samples (global model) ----
            client_g, client_h = {}, {}
            for cid in client_ids:
                p = softmax(client_score[cid])
                g = p.copy()
                g[np.arange(sizes[cid]), client_yi[cid]] -= 1.0
                client_g[cid], client_h[cid] = g, p * (1.0 - p)

            # ---- local at the Builder: grow K tree structures on its own data ----
            b = choose_builder(t, client_ids, sizes, cfg.builder_rule, rng, cfg.min_builder_samples)
            self.builders.append(b)
            if b not in local_cuts:
                local_cuts[b] = CutPoints.compute(client_X[b], n_bins=cfg.n_bins)
            structures = []
            for k in range(K):
                local_tree = build_one_federated_tree(
                    {b: client_X[b]}, {b: client_g[b][:, k]}, {b: client_h[b][:, k]}, local_cuts[b],
                    max_depth=cfg.max_depth, lam=cfg.lam, gamma=cfg.gamma, learning_rate=cfg.learning_rate,
                    min_child_weight=cfg.min_child_weight,
                    comm_tracker={"upload_mb": 0.0, "download_mb": 0.0},   # single client: nothing is sent
                )
                structures.append(structure_only(local_tree))

            # ---- communication 1: Builder -> other clients (tree structures) ----
            self.comm_rounds += 1
            self._send(structures, "structure_mb", n_receivers=M - 1)

            # ---- communication 2: every client -> aggregator (per-leaf sums of g and h) ----
            self.comm_rounds += 1
            n_leaves = [len(collect_leaves(s)) for s in structures]
            total = [np.zeros((n_leaves[k], 2)) for k in range(K)]
            client_leaf_idx = {}
            for cid in client_ids:
                stats, idx_k = [], []
                for k in range(K):
                    leaf_idx = route_to_leaves(structures[k], client_X[cid])
                    idx_k.append(leaf_idx)
                    G = np.bincount(leaf_idx, weights=client_g[cid][:, k], minlength=n_leaves[k])
                    H = np.bincount(leaf_idx, weights=client_h[cid][:, k], minlength=n_leaves[k])
                    stats.append(np.stack([G, H], axis=1))
                client_leaf_idx[cid] = idx_k
                for k, s in enumerate(self._send(stats, "leaf_stats_mb")):
                    total[k] += s

            # ---- global at the aggregator: leaf weights from the summed statistics ----
            leaf_values = [np.array([leaf_value(G, H, cfg.lam) * cfg.learning_rate for G, H in total[k]])
                           for k in range(K)]

            # ---- communication 3: aggregator -> all clients (leaf values) ----
            self.comm_rounds += 1
            self._send(leaf_values, "leaf_values_mb", n_receivers=M)
            round_trees = []
            for k in range(K):
                tree = copy.deepcopy(structures[k])
                for leaf, v in zip(collect_leaves(tree), leaf_values[k]):
                    leaf.value = float(v)
                round_trees.append(tree)
                for cid in client_ids:                       # clients update their own scores
                    client_score[cid][:, k] += leaf_values[k][client_leaf_idx[cid][k]]
            self.trees.append(round_trees)

            if (t + 1) % 10 == 0:
                print(f"  Boosting round {t + 1}/{cfg.n_trees} done (builder = client {b})")
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        scores = np.zeros((X.shape[0], len(self.classes_)))
        for round_trees in self.trees:
            for k, tree in enumerate(round_trees):
                scores[:, k] += predict_tree(tree, X)
        return self.classes_[np.argmax(scores, axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        y_pred = self.predict(X)
        return {
            "accuracy": accuracy_score(y, y_pred) * 100,
            "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0) * 100,
            "f1_weighted": f1_score(y, y_pred, average="weighted", zero_division=0) * 100,
        }

# Full pipeline

In [ ]:
ALL_RESULTS = []      # one summary row per run (saved to eflboost_results.csv at the end)


def run_eflboost(
    df: pd.DataFrame,
    dataset_name: str,
    label_col: str = "Label",
    partition_mode: str = "iid",          # "iid" | "dirichlet"
    n_clients: int = 10,
    dirichlet_alpha: float = 0.5,
    n_trees: int = 50,
    max_depth: int = 6,
    n_bins: int = 64,
    learning_rate: float = 0.1,
    builder_rule: str = "round_robin",
    test_size: float = 0.20,
    random_state: int = 42,
    verbose: bool = True,
) -> Dict:

    feature_cols = [c for c in df.columns if c != label_col]
    X = df[feature_cols].values.astype(np.float64)
    y = df[label_col].values

    # Global train/test
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Partition
    if partition_mode == "iid":
        client_data = partition_iid(X_tr, y_tr, n_clients, seed=random_state)
    else:
        client_data = partition_dirichlet(X_tr, y_tr, n_clients, alpha=dirichlet_alpha, seed=random_state)

    if verbose:
        print(f"Dataset: {dataset_name} {df.shape} | mode={partition_mode} | clients={n_clients}")
        for cid, (Xc, yc) in client_data.items():
            print(f"  Client {cid}: {len(yc)} samples, classes={dict(Counter(yc))}")

    cfg = EFLBoostConfig(n_trees=n_trees, max_depth=max_depth, n_bins=n_bins, learning_rate=learning_rate,
                         random_state=random_state, builder_rule=builder_rule)

    # ----- Federated training -----
    t0 = time.perf_counter()
    model = EFLBoost(cfg)
    model.fit(client_data)
    train_time = time.perf_counter() - t0

    metrics = model.score(X_te, y_te)
    inf_stats = measure_inference_time(model.predict, X_te)
    model_size_mb = sizeof_mb(model.trees)
    n_tree_objs = sum(len(r) for r in model.trees)
    n_nodes = sum(count_nodes(t) for r in model.trees for t in r)
    total_comm_mb = sum(model.comm_mb.values())
    y_pred = model.predict(X_te)

    results = {
        "metrics": metrics,
        "efficiency": {
            "n_clients": n_clients,
            "n_boosting_rounds": n_trees,
            "max_depth": max_depth,
            "train_time_s": train_time,
            "test_time_total_s": inf_stats["test_time_total_s"],
            "test_time_per_sample_ms": inf_stats["test_time_per_sample_ms"],
            "model_size_mb": model_size_mb,
            "n_trees": n_tree_objs,
            "n_nodes": n_nodes,
            "n_comm_rounds": model.comm_rounds,
            "n_messages": model.n_messages,
            **model.comm_mb,
            "total_comm_mb": total_comm_mb,
        },
        "builders": model.builders,
        "model": model,
        "y_test": y_te,
        "y_pred": y_pred,
        "classification_report": classification_report(y_te, y_pred, zero_division=0, output_dict=True),
    }
    ALL_RESULTS.append({"dataset": dataset_name, "partition": partition_mode,
                        "alpha": dirichlet_alpha if partition_mode == "dirichlet" else None,
                        **metrics, **results["efficiency"]})

    if verbose:
        e = results["efficiency"]
        print("\n===== RESULTS =====")
        print(f"Accuracy     : {metrics['accuracy']:.2f}%")
        print(f"F1-macro     : {metrics['f1_macro']:.2f}%")
        print(f"F1-weighted  : {metrics['f1_weighted']:.2f}%")
        print("\n===== EFFICIENCY =====")
        print(f"Train time          : {train_time:.1f} s")
        print(f"Test time (total)   : {e['test_time_total_s']:.4f} s")
        print(f"Test time / sample  : {e['test_time_per_sample_ms']:.4f} ms")
        print(f"Model size          : {model_size_mb:.3f} MB | {n_tree_objs} trees | {n_nodes} nodes")
        print(f"Total communication : {total_comm_mb:.2f} MB  (structures {e['structure_mb']:.2f} | "
              f"leaf stats {e['leaf_stats_mb']:.2f} | leaf values {e['leaf_values_mb']:.2f})")
        print(f"Comm rounds         : {model.comm_rounds}  ({model.n_messages} messages)")
        print(f"Builders per round  : {dict(sorted(Counter(model.builders).items()))}")
        print("\nClassification Report:")
        print(classification_report(y_te, y_pred, zero_division=0))

    return results

# Experiments

# 1. CICIDS2017

In [ ]:
df = pd.read_csv("CICIDS2017_sample_km.csv")

# IID

In [ ]:
# ----- IID -----
results_cic_iid = run_eflboost(
    df,
    dataset_name="CICIDS2017",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

# Non-IID

In [ ]:
# ----- Non-IID (Dirichlet) -----
results_cic_noniid = run_eflboost(
    df,
    dataset_name="CICIDS2017",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,      # smaller = stronger heterogeneity
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

# 2. IoTID2020

In [ ]:
df = pd.read_csv("IoT_Network_Intrusion_Dataset_0.1.csv")

In [ ]:
le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))
# No MinMaxScaler: trees only compare thresholds, so scaling does not change the model.

# IID

In [ ]:
# ----- IID -----
results_iot_iid = run_eflboost(
    df,
    dataset_name="IoTID2020",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

# Non-IID

In [ ]:
# ----- Non-IID (Dirichlet) -----
results_iot_noniid = run_eflboost(
    df,
    dataset_name="IoTID2020",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

# 3. CICEVSE2024

In [ ]:
df = pd.read_csv("CICEVSE2024_12classes_sample.csv")

# IID

In [ ]:
# ----- IID -----
results_evse_iid = run_eflboost(
    df,
    dataset_name="CICEVSE2024",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

# Non-IID

In [ ]:
# ----- Non-IID (Dirichlet) -----
results_evse_noniid = run_eflboost(
    df,
    dataset_name="CICEVSE2024",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

# Summary of all runs

In [ ]:
summary = pd.DataFrame(ALL_RESULTS)
summary.to_csv("eflboost_results.csv", index=False)
cols = ["dataset", "partition", "accuracy", "f1_macro", "f1_weighted", "train_time_s", "test_time_per_sample_ms",
        "model_size_mb", "n_trees", "n_nodes", "n_comm_rounds", "total_comm_mb"]
display(summary[cols].round(4))